# 9장 검색과 RAG의 평가와 개선

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `09_검색과_RAG의_평가와_개선.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 터미널에서 실행하고 커널을 다시 시작합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(공개 LLM, 임베딩 모델, 리랭커)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |
| `EMBED_MODEL` | 설정하지 않아도 됩니다. 다른 임베딩 모델을 쓰거나 모델 파일이 든 폴더를 따로 받았을 때, 모델 이름이나 폴더 경로를 넣습니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정한 값은 모든 장의 노트북에 그대로 적용됩니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install sentence-transformers rank-bm25 pypdf
# 9장과 관통 프로젝트 4단계에서 씁니다. 8장의 준비에 평가 함수와 리랭커, 채점 프롬프트를 더했습니다
import csv
import random
import re

import numpy as np

import corpus
import embed

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 관통 프로젝트 1, 2단계의 코드가 씁니다
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']

# 100만 토큰당 단가(달러). 5장과 같습니다(llm.py가 쓰는 Anthropic 모델, 2026-11 확인)
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


catalog = corpus.catalog()                               # {문서 id: 메타데이터}. 문서 120건
questions = {item['id']: item for item in corpus.evalset()}   # 평가 질문 62개
chunks, failed = corpus.load_chunks()                    # 문서마다 파싱하고 절 단위 청크로 나눕니다(corpus.py)
searcher = corpus.Search(chunks)                         # 청크마다 임베딩과 BM25 인덱스를 만듭니다. 1~2분 걸립니다


def is_valid(chunk):
    """메타데이터 필터. 폐기된 구판의 청크를 검색 대상에서 뺍니다"""
    return chunk['status'] == '유효'


RAG_PROMPT = """아래 [검색 결과]만 근거로 질문에 답하세요.
- 문장마다 근거가 된 청크의 번호를 [SOP-WB-001#3]처럼 적습니다.
- 검색 결과에 답이 없으면 "검색한 문서에서 찾지 못했습니다"라고만 답합니다.
- 검색 결과끼리 내용이 다르면 날짜가 늦은 문서를 따르고, 내용이 다르다는 것을 함께 적습니다.

[검색 결과]
{context}

[질문]
{question}"""


def context_of(hits, with_meta=True):
    """검색한 청크를 번호와 함께 이어 붙인 글. with_meta가 참이면 청크마다 상태, 개정, 날짜를 붙입니다"""
    blocks = []
    for hit in hits:
        meta = f" (상태 {hit['status']}, 개정 {hit['revision']}, {hit['date']})" if with_meta else ''
        blocks.append(f"[{hit['chunk_id']}]{meta}\n{hit['text']}")
    return '\n\n'.join(blocks)


def rag_prompt(question, hits, with_meta=True):
    """8장의 RAG 프롬프트. 검색한 청크를 RAG_PROMPT에 넣습니다"""
    return RAG_PROMPT.format(context=context_of(hits, with_meta), question=question)


def docs_of(indices):
    """청크 번호의 순위를 문서 id의 순위로 바꿉니다. 한 문서의 청크가 여러 번 나오면 처음 나온 순위만 남깁니다"""
    order = []
    for index in indices:
        if chunks[index]['doc_id'] not in order:
            order.append(chunks[index]['doc_id'])
    return order


def first_hit(order, gold):
    """문서 순위에서 정답 문서가 처음 나오는 순위. 없으면 None"""
    return next((rank for rank, doc_id in enumerate(order, 1) if doc_id in gold), None)


def load_reranker():
    """예제 9-2용. 크로스 인코더 리랭커를 PC에서 불러옵니다. 처음 한 번 수백 MB를 내려받습니다"""
    from sentence_transformers import CrossEncoder
    name = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
    embed.ask_endpoint(name)                     # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
    return CrossEncoder(name)


def rag_answer(question, k=4):
    """8장의 RAG. 유효한 문서에서 청크 k개를 검색해 RAG_PROMPT로 묻고 (응답, 검색한 청크)를 돌려줍니다"""
    hits = searcher.search(question, k=k, where=is_valid)
    return llm.ask(rag_prompt(question, hits), model='소형', max_tokens=2000), hits


JUDGE_PROMPT = """너는 사내 문서 질의응답 시스템의 채점자다. [답]을 아래 두 기준으로 따로 채점한다.
- faithful: [답]의 내용이 모두 [검색 결과]에서 확인되면 true, 검색 결과에 없는 내용이 하나라도 있으면 false.
- correct: [답]이 [기준 답]과 같은 사실을 말하면 true. 기준 답이 "문서에 답이 없다"이면, 찾지 못했다고 답했을 때만 true.
reason에는 판정의 근거를 한 문장으로 쓴다.

[검색 결과]
{context}

[질문]
{question}

[기준 답]
{reference}

[답]
{answer}"""
JUDGE_SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['faithful', 'correct', 'reason'],
                'properties': {'faithful': {'type': 'boolean'}, 'correct': {'type': 'boolean'}, 'reason': {'type': 'string'}}}


print(f'문서 {len(catalog)}건, 청크 {len(chunks)}개, 글을 뽑지 못한 문서 {failed}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. "준비 (1)" 셀을 다시 실행해 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `ModuleNotFoundError: No module named '...'` | 패키지를 설치하지 않았습니다. "준비 (2)" 셀의 `# 설치:` 주석에 있는 명령을 터미널에서 실행하고 커널을 다시 시작합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 9-1. 두 검색의 recall@k와 MRR

교재의 절: "검색 지표"

정답 문서가 있는 평가 질문 54개를 임베딩 검색과 키워드 검색으로 찾아, 문서 순위로 바꾼 뒤 recall@1, recall@3, recall@5, MRR을 계산합니다. 7장 예제 7-2는 질문 16개로 1위만 셌습니다. 질문 전체에서 두 검색의 차이가 얼마인지, 어느 지표에서 차이가 나는지 봅니다. API를 호출하지 않습니다.

In [ ]:
answerable = [item for item in questions.values() if item['docs']]          # 답이 없는 질문 8개는 뺍니다


def ranks_by(method):
    """질문마다 정답 문서가 처음 나온 문서 순위. 찾지 못하면 None"""
    return [first_hit(docs_of(searcher.ranking(item['question'], method, where=is_valid)), item['docs'])
            for item in answerable]


for method in ['embedding', 'keyword']:
    ranks = ranks_by(method)
    recall = {k: sum(rank is not None and rank <= k for rank in ranks) / len(ranks) for k in (1, 3, 5)}
    mrr = sum(1 / rank for rank in ranks if rank) / len(ranks)
    print(f'{method}: recall@1 {recall[1]:.2f}, recall@3 {recall[3]:.2f}, recall@5 {recall[5]:.2f}, MRR {mrr:.3f}')
    print('  5위 안에 없는 질문:', [item['id'] for item, rank in zip(answerable, ranks) if rank is None or rank > 5])

embedding: recall@1 0.70, recall@3 0.89, recall@5 0.91, MRR 0.799
  5위 안에 없는 질문: ['E40', 'E45', 'E46', 'E47', 'E48']
keyword: recall@1 0.67, recall@3 0.87, recall@5 0.93, MRR 0.775
  5위 안에 없는 질문: ['E23', 'E45', 'E47', 'E48']


## 예제 9-2. 하이브리드 검색과 리랭킹

교재의 절: "리랭킹"

예제 9-1의 질문 54개를 하이브리드 검색(RRF)으로 찾고, 하이브리드 검색의 상위 청크 10개를 크로스 인코더로 다시 줄 세웁니다. 두 방법의 recall@1과 MRR, 질문 하나에 걸린 시간을 예제 9-1의 결과와 비교합니다. 예제 9-1을 먼저 실행합니다. 리랭커를 처음 불러올 때 모델을 내려받고, 리랭킹은 몇 분 걸립니다. API를 호출하지 않습니다.

In [ ]:
reranker = load_reranker()


def reranked(question, candidates=10):
    """하이브리드 검색의 상위 청크 candidates개를 크로스 인코더의 점수 순으로 다시 줄 세웁니다"""
    top = searcher.ranking(question, 'hybrid', where=is_valid)
    scores = reranker.predict([(question, chunks[index]['text']) for index in top[:candidates]])
    return [top[index] for index in np.argsort(-scores)] + top[candidates:]


for name, ranking in [('hybrid', lambda question: searcher.ranking(question, 'hybrid', where=is_valid)),
                      ('hybrid + rerank', reranked)]:
    started = time.time()
    ranks = [first_hit(docs_of(ranking(item['question'])), item['docs']) for item in answerable]
    print(f'{name}: recall@1 {sum(rank == 1 for rank in ranks) / len(ranks):.2f}, '
          f'MRR {sum(1 / rank for rank in ranks if rank) / len(ranks):.3f}, 질문당 {(time.time() - started) / len(ranks):.2f}초')
    print('  1위를 놓친 질문:', [(item['id'], rank) for item, rank in zip(answerable, ranks) if rank != 1])

hybrid: recall@1 0.72, MRR 0.806, 질문당 0.19초
  1위를 놓친 질문: [('E13', 3), ('E19', 2), ('E23', 4), ('E25', 2), ('E36', 2), ('E37', 2), ('E40', 4), ('E41', 3), ('E44', 4), ('E45', 23), ('E46', 4), ('E47', None), ('E48', None), ('E50', 3), ('E53', 2)]
hybrid + rerank: recall@1 0.81, MRR 0.874, 질문당 2.45초
  1위를 놓친 질문: [('E23', 2), ('E34', 2), ('E40', 3), ('E41', 3), ('E44', 2), ('E45', 23), ('E46', 2), ('E47', None), ('E48', None), ('E50', 2)]


## 예제 9-3. 정답성과 충실도 채점하기

교재의 절: "답 평가"

평가 질문 8개(답이 없는 질문과 스캔 문서에만 답이 있는 질문 포함)를 8장의 RAG로 답하고, 같은 소형 모델로 정답성과 충실도를 채점합니다. 검색 결과에 정답 문서가 있었는지(찾음), 핵심어가 답에 있는지(`key`)도 함께 출력합니다. 채점 모델의 판정과 핵심어 비교가 다른 질문이 있는지 봅니다. 이 셀은 API를 16번 호출합니다.

In [ ]:
judged_rows = []
for qid in ['E06', 'E13', 'E23', 'E31', 'E41', 'E45', 'E47', 'E55']:
    item = questions[qid]
    reply, hits = rag_answer(item['question'])
    verdict = json.loads(llm.ask(JUDGE_PROMPT.format(context=context_of(hits), question=item['question'], reference=item['answer'],
                                                     answer=reply.text), model='소형', max_tokens=2000, schema=JUDGE_SCHEMA).text)
    found = any(hit['doc_id'] in item['docs'] for hit in hits) if item['docs'] else None
    has_key = all(key in reply.text for key in item['key']) if item['key'] else '찾지 못했습니다' in reply.text
    judged_rows.append({'id': qid, 'type': item['type'], 'found': found, 'key': has_key, **verdict})
    print(f"{qid}({item['type']}) 찾음 {found}, 핵심어 {has_key}, 정답성 {verdict['correct']}, 충실도 {verdict['faithful']}")
    print(f"  답: {reply.text.strip().replace(chr(10), ' ')[:110]}\n  채점 근거: {verdict['reason'][:110]}")

E06(사실) 찾음 True, 핵심어 True, 정답성 True, 충실도 True
  답: 로트 수율이 **97% 미만**이면 로트를 홀드하고 테스트기술팀과 품질보증팀에 알립니다 [SOP-PT-001#4].
  채점 근거: 답의 내용인 97% 미만 시 로트 홀드 및 테스트기술팀·품질보증팀 통보는 SOP-PT-001#4 검색 결과에 그대로 있고, 기준 답과 같은 사실이다.
E13(바꿔 말함) 찾음 True, 핵심어 True, 정답성 True, 충실도 True
  답: 검색 결과에 따르면, 다이 뒷면 가장자리 칩핑의 기준은 **40μm 이하**입니다[QIR-2025-044#2]. 한 사례에서는 62μm가 측정되어 이 기준을 넘었고, 불량은 웨이퍼 후반부 절단 줄에
  채점 근거: 답의 40μm 이하 기준, 62μm 측정, 후반부 절단 줄 집중, SOP의 수치 없는 절차 내용이 모두 검색 결과에 있고, 핵심 수치 40μm 이하가 기준 답과 일치한다.
E23(코드) 찾음 True, 핵심어 False, 정답성 False, 충실도 True
  답: 검색한 문서에서 찾지 못했습니다
  채점 근거: [답]은 '찾지 못했다'는 내용뿐이라 검색 결과에 없는 사실을 덧붙이지 않아 faithful하지만, [기준 답]은 플라즈마 세정 후 9시간 경과로 패드에 유기물이 재부착됐다는 구체적 원인이므로 문서
E31(개정) 찾음 True, 핵심어 True, 정답성 True, 충실도 True
  답: QX48의 리드 코플래너리티는 **0.08mm를 초과하면 불량**입니다 [SOP-VI-001#3]. 리드 코플래너리티는 패키지를 평면에 놓았을 때 가장 높이 뜬 리드와 평면 사이의 거리입니다 [SO
  채점 근거: 답의 0.08mm 초과 불량 기준과 리드 코플래너리티 정의는 SOP-VI-001#3에 있고, 회의록의 0.08mm 요구와 SOP가 회의록보다 늦은 날짜라는 설명도 검색 결과로 뒷받침되며, 기준 답과
E41(충돌) 찾음 True, 핵심어 True, 정답성 True, 충실도 True
  

## 실습 9-1. 질의 재작성

교재의 절: "개념 확인 실습"

하이브리드 검색에서 정답 문서가 1위에 오지 못한 질문 세 개(E13, E23, E44)를 LLM으로 검색어 세 개로 바꾸고, 원래 질문과 검색어 세 개의 검색 결과를 RRF로 합칩니다. `REWRITE_PROMPT`에 LLM에게 줄 지시를 작성하고, 질문마다 정답 문서의 순위가 오를지 `my_guess`에 먼저 적은 뒤 실행합니다. 순위가 오르거나 내린 이유를 LLM이 만든 검색어로 설명하면 끝납니다. 이 셀은 API를 3번 호출합니다.

In [ ]:
REWRITE_PROMPT = ''  # TODO: {question}을 사내 문서 검색에 쓸 검색어 세 개로 바꾸라는 지시. 한 줄에 하나씩만 출력하게 합니다
my_guess = {'E13': None, 'E23': None, 'E44': None}  # TODO: 정답 문서의 순위가 오를지('오름', '같음', '내림')

for qid in ['E13', 'E23', 'E44']:
    item = questions[qid]
    reply = llm.ask(REWRITE_PROMPT.format(question=item['question']), model='소형', max_tokens=2000)
    queries = [line.strip('-• ').strip() for line in reply.text.splitlines() if line.strip()][:3]
    before = docs_of(searcher.ranking(item['question'], 'hybrid', where=is_valid))
    after = docs_of(corpus.rrf([searcher.ranking(query, 'hybrid', where=is_valid) for query in [item['question']] + queries]))
    print(f"{qid} {item['question']}\n  검색어: {queries}")
    print(f"  정답 문서 순위 {first_hit(before, item['docs'])}위 → {first_hit(after, item['docs'])}위, 예측 {my_guess[qid]}")